In [ ]:
# Download data

In [ ]:
import os
import datasets
import pandas as pd
import plotly.express as px
from tqdm import tqdm

In [ ]:
import plotly.io as pio
pio.renderers.default = 'notebook' 

In [ ]:
conversation_metadata_fields = ['language', 'redacted', 'toxic', 'rate', 'title', 'custom_instruction', 'status',
                                'redacted']
user_metadata_fields = ['location', 'age', 'gender']

In [ ]:
# Get the CACHE_ROOT path from the environment variable you just set in Linux
CACHE_ROOT = os.environ.get("HF_HOME")

if CACHE_ROOT is None:
    # This should not happen, but a safe fallback in case the variable was lost
    CACHE_ROOT = "/cs/labs/oabend/tomer.shahaf/hf_cache_root"
    os.environ["HF_HOME"] = CACHE_ROOT

print(f"Loading dataset, processed cache will be in: {os.path.join(CACHE_ROOT, 'datasets')}")

# Load the dataset, explicitly setting the cache_dir argument
# This ensures both the raw download and the final processed files use the large disk.
ours = datasets.load_dataset(
    "shachardon/ShareLM", 
    split="train", 
    cache_dir=os.path.join(CACHE_ROOT, "datasets") 
)

print("Dataset loaded successfully!")

In [ ]:
ours = datasets.load_dataset(
    "shachardon/ShareLM", 
    split="train", 
    cache_dir=os.path.join(CACHE_ROOT, "datasets") 
)

In [ ]:
ours.shape

In [ ]:
lmsys_dataset = datasets.load_dataset("lmsys/lmsys-chat-1m", token=user_token)
lmsys_dataset_train = lmsys_dataset["train"]

In [ ]:
examples = []
for i in tqdm(range(lmsys_dataset_train.shape[0])):
    data = lmsys_dataset_train[i]
    conv = data["conversation"]
    user_metadata = {item: "" for item in user_metadata_fields}
    conversation_metadata = {"language": data["language"], "redacted": str(data["redacted"])}
    for field in conversation_metadata_fields:
        if field not in conversation_metadata:
            conversation_metadata[field] = ""
    example = {"conversation_id": data["conversation_id"], "conversation": conv,
               "source": "https://huggingface.co/datasets/lmsys/lmsys-chat-1m", "model_name": data["model"],
               "user_id": "", "user_metadata": user_metadata, "timestamp": "", "conversation_metadata":
                   conversation_metadata}
    examples.append(example)

lmsys_formatted_dataset = datasets.Dataset.from_pandas(pd.DataFrame(data=examples))

In [ ]:
wildchat_dataset = datasets.load_dataset("allenai/WildChat-1M", token=user_token)
wildchat_dataset_train = wildchat_dataset["train"]

In [ ]:
examples = []
for i in tqdm(range(wildchat_dataset_train.shape[0])):
    data = wildchat_dataset_train[i]
    conv = data["conversation"]
    user_metadata = {"location": f"{data['state']},{data['country']}"}
    conversation_metadata = {"language": data["language"], "redacted": str(data["redacted"]), "toxic": str(data["toxic"])}
    for field in conversation_metadata_fields:
        if field not in conversation_metadata:
            conversation_metadata[field] = ""
    for field in user_metadata_fields:
        if field not in user_metadata:
            user_metadata[field] = ""
    example = {"conversation_id": data["conversation_hash"], "conversation": conv,
               "source": "https://huggingface.co/datasets/allenai/WildChat-1M", "model_name": data["model"],
               "user_id": data["hashed_ip"], "user_metadata": user_metadata,
               "timestamp": data["timestamp"], "conversation_metadata": conversation_metadata}
    examples.append(example)

wildchat_formatted_dataset = datasets.Dataset.from_pandas(pd.DataFrame(data=examples))

In [ ]:
dataset_all = datasets.concatenate_datasets([ours, lmsys_formatted_dataset, wildchat_formatted_dataset])

In [ ]:
dataset_all_df = dataset_all.to_pandas()
dataset_all_df.to_parquet("merged_dataset.parquet", index=False)

In [ ]:
# ShareLM EDA

In [ ]:
# from collections import Counter
# import pandas as pd
# import plotly.express as px

# # 1️⃣ Count occurrences directly from HF Dataset
# counts = Counter(ours["source"])  # 'ours' is your HF Dataset object

# # 2️⃣ Convert counts dictionary to pandas DataFrame
# model_counts = pd.DataFrame( 
#     list(counts.items()),
#     columns=["model_name", "count"]
# ).sort_values("count", ascending=False)

# # 3️⃣ Keep top N models, group the rest as 'Other'
# N = 10
# top_10_models = model_counts.head(N)
# other_models = model_counts.iloc[N:]
# other_count = other_models['count'].sum()

# if other_count > 0:
#     other_row = pd.DataFrame([{'model_name': 'Other', 'count': other_count}])
#     plot_df = pd.concat([top_10_models, other_row], ignore_index=True)
# else:
#     plot_df = top_10_models

# # 4️⃣ Plot pie chart
# fig = px.pie(
#     plot_df,
#     values='count',
#     names='model_name',
#     hole=.3
# )

# fig.update_layout(
#     title={
#         'text': f'Top {N} model names',
#         'font': {'size': 14}
#     }
# )
# fig.show()


In [ ]:
# # start with plugin data
# SHARELM_PLUGIN_SOURCE = "https://chromewebstore.google.com/detail/sharelm-share-your-chat-c/nldoebkdaiidhceaphmipeclmlcbljmh" 
# sharelm_dataset = ours.filter(lambda x: x["source"] == SHARELM_PLUGIN_SOURCE)
# sharelm_dataset.shape

In [ ]:
# sharelm_dataset.to_parquet("sharelm_dataset.pqt")

sharelm_dataset = datasets.load_dataset(
    "parquet", 
    data_files="sharelm_dataset.pqt"
)
sharelm_dataset.shape

In [ ]:
share_lm_df = sharelm_dataset["train"].to_pandas()

In [ ]:
model_counts = share_lm_df['model_name'].value_counts().reset_index()
model_counts.columns = ['model_name', 'count']

N = 10
top_10_models = model_counts.head(N)
other_models = model_counts.iloc[N:]

# 3. Calculate the 'Other' count and combine
other_count = other_models['count'].sum()
if other_count > 0:
    other_row = pd.DataFrame([{'model name': 'Other', 'count': other_count}])
    plot_df = pd.concat([top_10_models, other_row], ignore_index=True)
else:
    plot_df = top_10_models

fig = px.pie(
    plot_df,
    values='count',
    names='model_name',
    hole=.3)

fig.update_layout(
    title={
        'text': f'Top {N} model names',
        'font': {'size': 14}
    }
)
fig.show()

In [ ]:
user_counts = share_lm_df['user_id'].value_counts().reset_index()
user_counts.columns = ['user_id', 'count']

N = 10
top_10_users = user_counts.head(N)
other_users = user_counts.iloc[N:]

# 3. Calculate the 'Other' count and combine
other_count = other_users['count'].sum()
if other_count > 0:
    other_row = pd.DataFrame([{'user name': 'Other', 'count': other_count}])
    plot_df = pd.concat([top_10_users, other_row], ignore_index=True)
else:
    plot_df = top_10_users

# 2. Create the Plotly Express pie chart
fig = px.pie(
    plot_df,
    values='count',
    names='user_id',
    hole=.3 # Optional: for a donut chart
)

fig.update_layout(
    title={
        'text': f'Top {N} user ids',
        'font': {'size': 14}
    }
)
fig.show()

In [ ]:
share_lm_df.timestamp

In [ ]:
import pandas as pd
DATE_FORMAT = '%Y-%m-%d %H:%M:%S.%f'

share_lm_df['timestamp'] = (share_lm_df['timestamp'].astype(str).str.strip().str.replace('Z', ''))
share_lm_df['timestamp'] = pd.to_datetime(
    share_lm_df['timestamp'],
    format=DATE_FORMAT,
    errors='coerce' 
)

share_lm_df['YearMonth'] = share_lm_df['timestamp'].dt.strftime('%Y-%m')

event_counts = share_lm_df['YearMonth'].value_counts().reset_index()
event_counts.columns = ['YearMonth', 'Count']
event_counts = event_counts.sort_values(by='YearMonth')

fig = px.bar(
    event_counts,
    x='YearMonth',
    y='Count',
    title='Count of Events by Year and Month',
    labels={'YearMonth': 'Month', 'Count': 'Event Count'},
    color_discrete_sequence=['#4c78a8']
)

fig.show()

In [ ]:
share_lm_df["conversation_len"] = share_lm_df.conversation.apply(lambda x: len(x))

In [ ]:
# Assuming you set X_MAX = 10 previously
X_MAX = 50

fig = px.histogram(
    share_lm_df,
    x='conversation_len',
    title='Distribution of Conversation Lengths',
    labels={'conversation_len': 'Conversation Length (turns)', 'count': 'Frequency (Count)'}
)

fig.update_traces(
    xbins=dict(
        start=0,
        size=1  
    ),
    marker_color='#1f77b4', 
    opacity=0.8
)
# ---------------------------------------

# Keep the focus on the 0-10 range as requested
fig.update_xaxes(
    range=[0, X_MAX],
    tickfont={'size': 8},
    title_font={'size': 12}
)
fig.update_layout(bargap=0.1)

fig.show()

In [ ]:
from tqdm import tqdm
tqdm.pandas()

In [ ]:
share_lm_df["user_prompts"] = share_lm_df["conversation"].progress_apply(
  lambda conversation: [turn["content"] for turn in conversation if turn["role"] == "user"])

In [ ]:
share_lm_df = share_lm_df[share_lm_df["user_prompts"].apply(lambda x: len(x) > 0)]
share_lm_df = share_lm_df[share_lm_df["user_prompts"].apply(lambda user_prompts: all([x is not None for x in user_prompts]))]
share_lm_df.shape

In [ ]:
import string

def is_valid_prompt(prompt: str) -> bool:
    allowed_chars = set(string.printable)
    if not prompt.strip():
      return False
    is_printable_prompt = all(ch in allowed_chars for ch in prompt)
    return is_printable_prompt

In [ ]:
share_lm_df["all_user_prompts_valid"] = share_lm_df["user_prompts"].progress_apply(
  lambda user_prompts: all([is_valid_prompt(prompt) for prompt in user_prompts]))
share_lm_df = share_lm_df[share_lm_df["all_user_prompts_valid"]]
share_lm_df.shape

In [ ]:
share_lm_df["user_prompts_count"] = share_lm_df["user_prompts"].apply(len)

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("intfloat/e5-small-v2")

In [ ]:
import torch

if torch.cuda.is_available():
    model.to('cuda')
    print("Model successfully moved to CUDA GPU.")
else:
    print("CUDA not available. Running on CPU.")

total_params = sum(p.numel() for p in model.parameters())
print(total_params)

In [ ]:
# user prompts

In [ ]:
MIN_THRESHOLD_CONVERSATION_LEN = 2
MAX_THRESHOLD_CONVERSATION_LEN = 10

share_lm_medium_conversations_df = share_lm_df[
    (share_lm_df["user_prompts_count"] >= MIN_THRESHOLD_CONVERSATION_LEN) &
     (share_lm_df["user_prompts_count"] <= MAX_THRESHOLD_CONVERSATION_LEN)]
share_lm_medium_conversations_df.shape

In [ ]:
share_lm_df = share_lm_medium_conversations_df.sample(1_000)
#share_lm_df = share_lm_medium_conversations_df
share_lm_df.shape

In [ ]:
share_lm_df["not_all_same_prompt"] = share_lm_df["user_prompts"].progress_apply(lambda user_prompts: all([prompt != user_prompts[0] for prompt in user_prompts[1:]]))
share_lm_df = share_lm_df[share_lm_df["not_all_same_prompt"]]
share_lm_df.shape

In [ ]:
share_lm_df_flat = share_lm_df.explode("user_prompts")
share_lm_df_flat = share_lm_df_flat.rename(columns={'user_prompts': 'user_prompt'})
share_lm_df_flat.shape

In [ ]:
share_lm_df_flat['encoded_text'] = 'query: ' + share_lm_df_flat['user_prompt']
embeddings = model.encode(
    share_lm_df_flat['encoded_text'].tolist(),
    #batch_size=512,
    batch_size=16,

    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)
share_lm_df_flat['user_prompts_embeddings'] = embeddings.cpu().numpy().tolist()

In [ ]:
import numpy as np
share_lm_df_vecotrs = share_lm_df_flat.groupby(level=0)['user_prompts_embeddings'].apply(list).reset_index(name='user_prompts_embeddings')

In [ ]:
share_lm_df_vecotrs["user_prompts_semantic_cosine_sim"] = share_lm_df_vecotrs["user_prompts_embeddings"].progress_apply(
    lambda user_prompts_embeddings: [round(np.dot(emb, user_prompts_embeddings[0]), 3) for emb in user_prompts_embeddings])

In [ ]:
share_lm_df = pd.merge(
    share_lm_df,
    share_lm_df_vecotrs[['index', 'user_prompts_embeddings', 'user_prompts_semantic_cosine_sim']],
    left_index=True,
    right_on='index',
    how='left'
)
share_lm_df.shape

In [ ]:
# ls drive/MyDrive/llms_huji
# share_lm_df.to_parquet("drive/MyDrive/llms_huji/share_lm_df_with_vectors.pqt")
#share_lm_df = pd.read_parquet("drive/MyDrive/llms_huji/share_lm_df_with_vectors.pqt")

In [ ]:
share_lm_df.head()

In [ ]:
conversation_id = "96a88019-31d6-42cf-a5c5-5e088acc87fd"
example = share_lm_df[share_lm_df["conversation_id"]==conversation_id]
print(example.conversation_id.values[0])
example.user_prompts.values[0]

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

df = share_lm_df.sample(50).explode('user_prompts_semantic_cosine_sim')
df['cosine_value'] = pd.to_numeric(df['user_prompts_semantic_cosine_sim'])
df['Rank'] = df.groupby(level=0).cumcount() + 1
df = df.reset_index()

fig = px.line(
    df,
    x='Rank',
    y='cosine_value',
    color='index',
    line_group='index',
    markers=True,
    title='Cosine similarity scores from all to first user prompts',
    hover_data={'index': True, 'Rank': True, 'cosine_value': ':.3f'}
)

fig.update_traces(mode='lines+markers')
fig.update_layout(
    xaxis_title='User prompt index',
    yaxis_title='Cosine similarity',
    legend_title='User Prompt Index',
    hovermode="closest"
)

fig.show()

In [ ]:
conversation_id = "35425"
example = share_lm_df[share_lm_df["index"]==92195]
print(example.conversation_id.values[0])
example.user_prompts.values[0]

In [ ]:
# top 10 median cosine conversations
share_lm_df["cosine_median"] = share_lm_df["user_prompts_semantic_cosine_sim"].apply(lambda x: np.median(x))
share_lm_df.sort_values(by="cosine_median", ascending=False).head(10)


In [ ]:
# top 10 mean cosine conversations
share_lm_df["cosine_mean"] = share_lm_df["user_prompts_semantic_cosine_sim"].apply(lambda x: np.mean(x))
share_lm_df.sort_values(by="cosine_mean", ascending=False).head(10)


In [ ]:
bin_settings = dict(
    start=-1,  # Start of the full range for calculation
    end=1,     # End of the full range for calculation
    size=0.01  # Bin width of 0.01
)

fig = px.histogram(
    share_lm_df,
    x='cosine_mean',
    nbins=10, # Optional: A hint for the number of bins (2/0.05 = 40)
    # The xbins setting ensures exact bin boundaries and size
    histnorm='percent',
    title='Distribution of Mean Cosine Similarity',
    labels={'cosine_mean': 'Mean Cosine Similarity Score', 'count': 'Frequency'},
    color_discrete_sequence=['#4C78A8']
)

# Explicitly apply the requested xbins to ensure perfect binning
fig.update_traces(
    xbins=bin_settings,
    marker=dict(line=dict(width=0.5, color='black')) # Added black border for better visibility of thin bars
)

# Customize Layout for clarity
fig.update_layout(
    xaxis_title='Mean Cosine Similarity (Bins of 0.01)',
    yaxis_title='Frequency (Count)', # UPDATED: Show raw counts
    bargap=0.005, # Adjusted gap for smaller bins
    template='plotly_white'
)
fig.show()

In [ ]:
SEMANTIC_CHANGE_COSINE_THRESHOLD = 0.85

def get_count_before_semantic_change(arr, threshold = SEMANTIC_CHANGE_COSINE_THRESHOLD):
  try:
    return next(i for i, x in enumerate(arr) if x < threshold)
  except StopIteration:
    return len(arr)

In [ ]:
share_lm_df["count_before_semantic_change"] = share_lm_df["user_prompts_semantic_cosine_sim"].apply(get_count_before_semantic_change)

In [ ]:
px.histogram(share_lm_df["count_before_semantic_change"])

In [ ]:
number_of_high_semantic_conversations = share_lm_df[share_lm_df["count_before_semantic_change"] >= MIN_THRESHOLD_CONVERSATION_LEN].shape[0]
print(f"number of high semantic conversations: {number_of_high_semantic_conversations}")
print(f"all with bigger then {SEMANTIC_CHANGE_COSINE_THRESHOLD} cosine and more at least the {MIN_THRESHOLD_CONVERSATION_LEN} user turns")

In [ ]:
# model answers

In [ ]:
share_lm_df["model_answers"] = share_lm_df["conversation"].progress_apply(
  lambda conversation: [turn["content"] for turn in conversation if turn["role"] == "assistant"])
share_lm_df["model_answers_count"] = share_lm_df["model_answers"].apply(len)

In [ ]:
share_lm_df_flat = share_lm_df.explode("model_answers")
share_lm_df_flat = share_lm_df_flat.rename(columns={'model_answers': 'model_answer'})
share_lm_df_flat.shape

In [ ]:
share_lm_df_flat['encoded_text'] = 'query: ' + share_lm_df_flat['model_answer']
embeddings = model.encode(
    share_lm_df_flat['encoded_text'].tolist(),
    #batch_size=512,
    batch_size=4,

    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)
share_lm_df_flat['model_answers_embeddings'] = embeddings.cpu().numpy().tolist()

In [ ]:
import numpy as np
share_lm_df_vecotrs = share_lm_df_flat.groupby(level=0)['model_answers_embeddings'].apply(list).reset_index(name='model_answers_embeddings')

In [ ]:
share_lm_df_vecotrs["model_answers_semantic_cosine_sim"] = share_lm_df_vecotrs["model_answers_embeddings"].progress_apply(
    lambda model_answers_embeddings: [round(np.dot(emb, model_answers_embeddings[0]), 3) for emb in model_answers_embeddings])

In [ ]:
share_lm_df_vecotrs

In [ ]:
share_lm_df_vecotrs.shape

In [ ]:
share_lm_df = pd.merge(
    share_lm_df,
    share_lm_df_vecotrs[['model_answers_embeddings', 'model_answers_semantic_cosine_sim']],
    right_index=True,
    left_index=True,
    how='left'
)
share_lm_df.shape

In [ ]:
share_lm_df.iloc[0]

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

df = share_lm_df.sample(50).explode('model_answers_semantic_cosine_sim')
df['cosine_value'] = pd.to_numeric(df['model_answers_semantic_cosine_sim'])
df['Rank'] = df.groupby(level=0).cumcount() + 1
df = df.reset_index()

fig = px.line(
    df,
    x='Rank',
    y='cosine_value',
    color='index',
    line_group='index',
    markers=True,
    title='Cosine similarity scores from all to first model answers',
    hover_data={'index': True, 'Rank': True, 'cosine_value': ':.3f'}
)

fig.update_traces(mode='lines+markers')
fig.update_layout(
    xaxis_title='Model answer index',
    yaxis_title='Cosine similarity',
    legend_title='Model answer Index',
    hovermode="closest"
)

fig.show()

In [ ]:
# top 10 mean cosine conversations
share_lm_df["model_cosine_mean"] = share_lm_df["model_answers_semantic_cosine_sim"].apply(lambda x: np.mean(x))

In [ ]:
bin_settings = dict(
    start=-1,  # Start of the full range for calculation
    end=1,     # End of the full range for calculation
    size=0.01  # Bin width of 0.01
)

fig = px.histogram(
    share_lm_df,
    x='model_cosine_mean',
    nbins=10, # Optional: A hint for the number of bins (2/0.05 = 40)
    # The xbins setting ensures exact bin boundaries and size
    histnorm='percent',
    title='Distribution of Mean Cosine Similarity',
    labels={'model_cosine_mean': 'Mean Model Answers Cosine Similarity Score', 'count': 'Frequency'},
    color_discrete_sequence=['#4C78A8']
)

# Explicitly apply the requested xbins to ensure perfect binning
fig.update_traces(
    xbins=bin_settings,
    marker=dict(line=dict(width=0.5, color='black')) # Added black border for better visibility of thin bars
)

# Customize Layout for clarity
fig.update_layout(
    xaxis_title='Mean Cosine Similarity (Bins of 0.01)',
    yaxis_title='Frequency (Count)', # UPDATED: Show raw counts
    bargap=0.005, # Adjusted gap for smaller bins
    template='plotly_white'
)
fig.show()

In [ ]:
SEMANTIC_MODEL_CHANGE_COSINE_THRESHOLD = 0.8

def get_count_before_model_semantic_change(arr, threshold = SEMANTIC_MODEL_CHANGE_COSINE_THRESHOLD):
  try:
    return next(i for i, x in enumerate(arr) if x < threshold)
  except StopIteration:
    return len(arr)

In [ ]:
share_lm_df["count_before_model_semantic_change"] = share_lm_df["model_answers_semantic_cosine_sim"].apply(get_count_before_model_semantic_change)
px.histogram(share_lm_df["count_before_model_semantic_change"])

In [ ]:
number_of_high_model_semantic_conversations = share_lm_df[share_lm_df["count_before_model_semantic_change"] >= MIN_THRESHOLD_CONVERSATION_LEN].shape[0]
print(f"number of high model semantic conversations: {number_of_high_model_semantic_conversations}")
print(f"all with bigger then {SEMANTIC_MODEL_CHANGE_COSINE_THRESHOLD} cosine and more at least the {MIN_THRESHOLD_CONVERSATION_LEN} user turns")

In [ ]:
final_df = share_lm_df[share_lm_df["count_before_model_semantic_change"] >= MIN_THRESHOLD_CONVERSATION_LEN]
sample = final_df.sample().iloc[0]
print(sample.conversation_id)
print(f"conversation length: {sample.conversation_len}")
print(sample.conversation)

In [ ]:
# use gpt3.5 to query about the prompts subject and so